# 01. Getting Started with VAFT

This 90-minute session introduces a VEST tokamak shot, its diagnostic data, and VAFT's public analysis and plotting interfaces. The reproducible core uses the packaged VEST sample for shot 39915; the optional lab branch reads a small selection from the public HSDS database when it has been configured.


## Session Overview

By the end of this session, you will be able to:

- describe a tokamak shot as a time-resolved experiment with multiple diagnostics;
- explain the roles of VAFT, OMAS, IMAS-style interface data structures (IDSs), and VEST data;
- load VAFT's packaged sample, inspect its available IDS roots and plot recipes, and make diagnostic plots through public APIs; and
- use the optional, read-only HSDS route without making the offline lesson depend on credentials.

Suggested timing: 15 minutes for context and data loading, 35 minutes for guided analysis, 15 minutes for the lab branch and interpretation checkpoints, and 25 minutes for the integrated analysis and independent exercise.

Run with `VAFT_TUTORIAL_MODE=offline` (the default) for the packaged sample. Set `VAFT_TUTORIAL_MODE=lab` only after configuring HSDS access with `hsconfigure`.


## Physical Context

A **shot** is one time-resolved tokamak discharge. Its trace data describe how programmed coils, magnetic fields, plasma current, and diagnostic signals evolve from prefill and breakdown through the plasma phase and shutdown. A diagnostic is not a result by itself: its signal needs a time reference, units, channel meaning, and physical interpretation.

VAFT organizes these data through OMAS objects using the IMAS vocabulary. At a practical level, an IDS is a named top-level data family such as `magnetics`, `pf_active`, `equilibrium`, or `spectrometer_uv`. This session treats that structure as a map: first find the relevant IDS, then inspect a documented path, then use a VAFT public plot recipe rather than rebuilding the plotting logic.

For the plasma-current trace below, look for the onset of current, a sustained interval, and the return toward zero. Those are operational phases to investigate, not universal labels that can be inferred from one plot alone.


## Load / Prepare Data

The setup cell chooses the execution mode, makes an ignored directory for figures created during your own run, and loads the packaged sample. It does not contact HSDS in offline mode.


In [ ]:
from pathlib import Path
import os

os.environ.setdefault("MPLBACKEND", "Agg")

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

import vaft

MODE = os.environ.get("VAFT_TUTORIAL_MODE", "offline").strip().lower()
if MODE not in {"offline", "lab"}:
    raise ValueError("VAFT_TUTORIAL_MODE must be 'offline' or 'lab'")

SHOT = 39915
OUTPUT_DIR = Path(
    os.environ.get("VAFT_TUTORIAL_OUTPUT_DIR", "tutorial/outputs/01")
).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ods = vaft.omas.sample_ods()
data_origin = "VAFT packaged sample"
print(f"Mode: {MODE}; source: {data_origin}; shot: {SHOT}")


In [ ]:
ids_roots = sorted(ods.keys())
print("Available IDS roots:")
for name in ids_roots:
    print(f"- {name}")

time_s = np.asarray(ods["magnetics.time"])
ip_a = np.asarray(ods["magnetics.ip.0.data"])
print(f"magnetics.time: {time_s.size} samples from {time_s.min():.4f} to {time_s.max():.4f} s")
print(f"magnetics.ip.0.data: {ip_a.size} samples; peak magnitude {np.abs(ip_a).max():.0f} A")


### Optional guided lab branch: read-only HSDS access

Set `VAFT_TUTORIAL_MODE=lab` only after configuring access. This branch deliberately reads only the `magnetics` IDS and retains the packaged sample for the remainder of the lesson, so every guided plot remains reproducible. A connection or credential problem is reported as a gated skip, not a lesson failure.


In [ ]:
lab_time_s = None
lab_ip_a = None

if MODE == "lab":
    try:
        with vaft.database.open(SHOT, source="public", paths="magnetics") as remote_ods:
            lab_time_s = np.asarray(remote_ods["magnetics.time"])
            lab_ip_a = np.asarray(remote_ods["magnetics.ip.0.data"])
        print(f"HSDS read succeeded: {lab_time_s.size} magnetic time samples.")
    except Exception as error:
        print("Lab extension skipped: HSDS read-only access is unavailable.")
        print(f"Reason: {type(error).__name__}: {error}")
else:
    print("Lab extension skipped in offline mode; the packaged sample remains active.")


## Guided Analysis

VAFT exposes plot recipes as public functions in `vaft.omas`. Discover the recipes that the current ODS supports before selecting one. The three guided plots move from a single operational trace to a magnetic overview and an impurity-spectrometer signal.


In [ ]:
available_plot_names = sorted(row["name"] for row in vaft.omas.available_plots(ods))
print(f"The packaged sample supports {len(available_plot_names)} VAFT plot recipes.")
print("Examples:", ", ".join(available_plot_names[:12]))


In [ ]:
fig_ip, ax_ip = vaft.omas.plot_magnetics_time_ip(ods, show=False)
ax_ip.set_title(f"VEST shot {SHOT}: plasma current")
fig_ip.savefig(OUTPUT_DIR / "session01_plasma_current.png", dpi=180, bbox_inches="tight")
display(fig_ip)


In [ ]:
fig_magnetics, axes_magnetics = vaft.omas.plot_magnetics_overview(
    ods, channels=[0, 1], show=False
)
fig_magnetics.savefig(OUTPUT_DIR / "session01_magnetic_overview.png", dpi=180, bbox_inches="tight")
display(fig_magnetics)


In [ ]:
fig_uv, axes_uv = vaft.omas.plot_spectrometer_uv_time_intensity(ods, show=False)
fig_uv.savefig(OUTPUT_DIR / "session01_uv_intensity.png", dpi=180, bbox_inches="tight")
display(fig_uv)


## Interpretation Checkpoints

Discuss these questions before moving on:

1. Which interval of the plasma-current trace would you investigate as the sustained plasma phase, and what evidence from the plot supports that choice?
2. Which magnetic or spectrometer signal changes near the current rise? Does that establish a causal relationship, or only a time correlation?
3. What additional metadata (units, channel position, calibration, or another diagnostic) would you need before making a stronger physical claim?

The aim is disciplined first-look interpretation: use the plots to form a question, then identify the information needed to answer it.


## Integrated Analysis

Select a public recipe from the available list, resolve its exported `vaft.omas.plot_*` function, and keep a small record of the choice. This is useful when the diagnostic is chosen at runtime while still calling the same public functions used in the guided analysis.


In [ ]:
selected_plot = "pf_active_time_current"
if selected_plot not in available_plot_names:
    raise ValueError(f"{selected_plot!r} is not available for this ODS")

selected_plot_function = getattr(vaft.omas, f"plot_{selected_plot}")
fig_selected, axes_selected = selected_plot_function(
    ods, channels=[0, 4, 5, 9], show=False
)
fig_selected.savefig(OUTPUT_DIR / f"session01_{selected_plot}.png", dpi=180, bbox_inches="tight")
display(fig_selected)


## Independent Exercise

Choose one diagnostic plot from `available_plot_names` that is different from the guided plasma-current plot. Change `exercise_plot`, run the cell, and write a three-sentence observation in a new Markdown cell below it:

1. identify the diagnostic and the time interval you examined;
2. describe one feature visible in the trace or image; and
3. state one additional measurement or metadata item needed to interpret that feature physically.

Good starting choices are `magnetics_time_diamagnetic_flux`, `pf_active_time_current`, or `spectrometer_uv_time_impurity`.


In [ ]:
exercise_plot = "magnetics_time_diamagnetic_flux"  # Replace with your chosen available recipe.
if exercise_plot not in available_plot_names:
    raise ValueError(
        f"Choose one of the {len(available_plot_names)} recipes listed in available_plot_names."
    )

exercise_plot_function = getattr(vaft.omas, f"plot_{exercise_plot}")
fig_exercise, axes_exercise = exercise_plot_function(ods, show=False)
fig_exercise.savefig(OUTPUT_DIR / f"session01_exercise_{exercise_plot}.png", dpi=180, bbox_inches="tight")
display(fig_exercise)


## Takeaways and Next Steps

You have used a packaged VEST ODS, inspected IDS roots and paths, discovered applicable VAFT plot recipes, and turned diagnostic data into questions rather than conclusions. The same offline-first pattern will continue through the course: start from a reproducible input, use a public VAFT API, then add a gated lab or solver extension only when the environment supports it.

For additional plotting examples, see [Plotting Sample Data with VAFT Plot Module](../notebooks/plotting_sample_using_vaft_plot_module.ipynb). Next, Session 02 connects these diagnostic traces to discharge operation and vacuum fields.

Run the code below in a final code cell when checking your own working copy; committed tutorial notebooks remain source-only and store no outputs.


In [ ]:
print(
    f"SESSION_01_OFFLINE_READY: shot={SHOT}; ids={len(ids_roots)}; "
    f"plots={len(available_plot_names)}; output_dir={OUTPUT_DIR}"
)
